In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

#df = pd.read_csv()

In [0]:
data_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/spx_with_indicators/"

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    DateType,
    DoubleType,
    StringType
)

In [0]:
df = spark.read.format("delta").load(data_path)

df = df.withColumn("Date", F.to_date("Date"))

In [0]:
#Here we are creating our entry signal to enter a long position based on the following conditions:
#- SMA_20 > SMA_50 > SMA_200
#- RSI_14 > 50

#Our exit signal is when the SMA_20 crosses below the SMA_50 
#SMA_20 <SMA_50

df_signal = (
    df.withColumn(
        "entry_signal",
        F.when(
            (F.col("SMA_20") > F.col("SMA_50")) & 
            (F.col("SMA_50") > F.col("SMA_200")) &
            (F.col("RSI_14") > F.lit(50)),
            F.lit(1)
        ).otherwise(F.lit(0))   
    )
    .withColumn(
        "exit_signal",
        F.when(F.col("SMA_20") < F.col("SMA_50"), F.lit(1)).otherwise(F.lit(0))
    )
)

In [0]:
trades_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/trades_MMAA_RSI_strategy_delta"

In [0]:
trades_schema = StructType([
    StructField("entry_date", DateType(), True),
    StructField("entry_price", DoubleType(), True),
    StructField("exit_date", DateType(), True),
    StructField("exit_price", DoubleType(), True),
    StructField("return", DoubleType(), True),
    StructField("mae", DoubleType(), True),
    StructField("reason", StringType(), True)
])

In [0]:
STOP_LOSS_PCT = 0.02
TAKE_PROFIT_PCT = 0.03

In [0]:
def backtest_one_series(pdf):
    pdf = pdf.sort_values("Date").reset_index(drop=True)

    trades = []
    in_pos = False
    entry_price = None
    entry_date = None
    mae = 0.0

    i = 0
    n = len(pdf)

    while i < n - 1:
        row = pdf.iloc[i]

        if not in_pos:
            if int(row["entry_signal"]) == 1:
                entry_row = pdf.iloc[i + 1]
                in_pos = True
                entry_price = float(entry_row["Open"])
                entry_date = entry_row["Date"]
                mae = 0.0
                i = i + i
                continue

        if in_pos:
            row = pdf.iloc[i]
            day_high = float(row["High"])
            day_low = float(row["Low"])
            day_close = float(row["Close"])

            mae = min(mae, (day_low - entry_price) / entry_price)

            stop_price = entry_price * (1 - STOP_LOSS_PCT)
            take_price = entry_price * (1 + TAKE_PROFIT_PCT)

            exit_reason = None
            exit_price = None
            exit_date = row["Date"]

            if day_low <= stop_price:
                exit_reason = "stop_loss"
                exit_price = stop_price

            elif day_high >= take_price:
                exit_reason = "take_profit"
                exit_price = take_price

            elif int(row["exit_signal"]) == 1:
                if i + 1 < n:
                    next_row = pdf.iloc[i+1]
                    exit_reason = "trend break"
                    exit_price = float(next_row["Open"])
                    exit_date = next_row["Date"]
                else: 
                    exit_reason = "end_of_data"
                    exit_price = day_close

            if exit_reason is not None:
                ret = (exit_price - entry_price / entry_price)
                trades.append({
                    "entry_date": entry_date,
                    "entry_price": entry_price,
                    "exit_date": exit_date,
                    "exit_price": exit_price,
                    "return": ret,
                    "mae": mae,
                    "reason": exit_reason
                })
                in_pos = False
                entry_price = None
                entry_date = None
                mae = 0.0

        i += 1
    return pd.DataFrame(trades)
                                              

In [0]:
df_one = df_signal.withColumn("series_id", F.lit("SPX"))

trades_spark = (
    df_one
    .select(
        "series_id",
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
        "SMA_20",
        "SMA_50",
        "SMA_200",
        "RSI_14",
        "entry_signal",
        "exit_signal"
    )
    .groupBy("series_id")
    .applyInPandas(backtest_one_series, schema = trades_schema)
)

In [0]:
trades_spark.write.format("delta").mode("overwrite").save(trades_path)

In [0]:
trades_spark.write.format("delta").mode("overwrite").saveAsTable("default.capstone_trades_MMAA_RSI_strategy")